In [ ]:
# Set your API key
from nnsight import LanguageModel, CONFIG
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import clear_output
import torch
import os
from dotenv import load_dotenv
load_dotenv()

CONFIG.set_default_api_key(os.getenv("NDIF_API_KEY"))
CONFIG.API.HOST = "https://api.ndif.us"
CONFIG.save()

# Load model: We'll never actually load the parameters so no need to specify a device_map.
#tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct") # Use float16 to save memory)
instruct_model = LanguageModel('meta-llama/Llama-3.3-70B-Instruct', device_map='auto',  dtype="bfloat16")
# llm = LanguageModel("EleutherAI/gpt-j-6b", device_map="auto")
clear_output()

# load in tokenizer
#tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-70B-Instruct")
#tokenizer.pad_token = None

print(instruct_model)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
decoder = model.lm_head

import torch
import matplotlib.pyplot as plt
import numpy as np

def visualize_decoder_matrix(decoder_matrix):
    """
    Visualize a decoder matrix with color mapping:
    - Largest positive value: dark blue
    - Zero: white
    - Largest negative value: red
    - Linear interpolation in between
    """
    # Convert to numpy for plotting
    matrix = decoder_matrix.detach().cpu().numpy()
    
    # Get the maximum absolute value for symmetric scaling
    max_val = torch.max(torch.abs(decoder_matrix)).item()
    
    # Create figure
    plt.figure(figsize=(12, 8))
    
    # Use RdBu_r colormap (Red-White-Blue reversed)
    # vmin and vmax center the colormap at 0
    im = plt.imshow(matrix, 
                    cmap='RdBu_r',  # Red for negative, Blue for positive
                    vmin=-max_val, 
                    vmax=max_val,
                    aspect='auto',
                    interpolation='nearest')
    
    # Add colorbar
    cbar = plt.colorbar(im)
    cbar.set_label('Weight Value', rotation=270, labelpad=20)
    
    # Labels
    plt.xlabel('Vocabulary Dimension')
    plt.ylabel('Hidden Dimension')
    plt.title(f'Decoder Matrix Visualization\nShape: {matrix.shape}')
    
    # Grid for better readability (optional, comment out for large matrices)
    if matrix.shape[0] < 50 and matrix.shape[1] < 50:
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"Matrix shape: {matrix.shape}")
    print(f"Max value: {max_val:.4f}")
    print(f"Min value: {torch.min(decoder_matrix).item():.4f}")
    print(f"Mean value: {torch.mean(decoder_matrix).item():.4f}")
    print(f"Std value: {torch.std(decoder_matrix).item():.4f}")

visualize_decoder_matrix(model.lm_head.weight)

NotImplementedError: Cannot copy out of meta tensor; no data!